In [4]:
import os
import sys
import tensorflow as tf
import numpy as np
import random

# Evitar ruidos innecesarios de TF
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.append("../")

from data_generatorHQoS import input_fn
from delay_model_HQoS import RouteNet_Fermi



# 2. CONFIGURACIÓN
PATH = "../datasets/oran-hqos-spwfq-sp_ALL/train"
LEARNING_RATE = 0.001  # Un poco más alto para depuración rápida
SCALE_FACTOR = 1e6     # Pasamos de segundos a microsegundos
SCALE_FACTOR_LABELS = 1  # No escalamos las etiquetas para ver el error real
# 3. CARGA DE DATOS (Tomamos solo 1 muestra para depurar)
print("[*] Cargando muestra de depuración...")
ds = input_fn(PATH, shuffle=False).take(1000)
features, labels = next(iter(ds))

# 4. INICIALIZACIÓN DEL MODELO Y Z-SCORE (Simulado para el test)
model = RouteNet_Fermi()

# Simulamos la estructura de z_score que requiere tu modelo
# Solo para que no de error al hacer el forward pass
if not hasattr(model, 'z_score'):
    model.z_score = {k: [0.0, 1.0] for k in [
        'traffic', 'packets', 'capacity', 'queue_size', 'eq_lambda'
    ]}

# 5. ESTRATEGIA DE DEPURACIÓN (Manual Training Loop)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_fn = tf.keras.losses.MeanAbsolutePercentageError()


for i in range(51):
    with tf.GradientTape() as tape:
        # Forward pass: El modelo predice en segundos
        raw_predictions = model(features, training=True)
        
        # Escalamos AMBOS a microsegundos para que el gradiente sea estable
        pred_scaled = raw_predictions * SCALE_FACTOR_LABELS
        labels_scaled = labels * SCALE_FACTOR_LABELS
        
        loss = loss_fn(labels_scaled, pred_scaled)
    
    # Cálculo de gradientes
    gradients = tape.gradient(loss, model.trainable_variables)
    
    # Aplicar gradientes
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    if i % 10 == 0:
        # Calculamos el gradiente medio para ver salud de la red
        grad_norm = np.mean([np.mean(np.abs(g)) for g in gradients if g is not None])
        print(f"Iteración {i:2d} | Loss: {loss.numpy():8.4f} | Grad Medio: {grad_norm:.8f}")

# 6. RESULTADO FINAL
print("\n--- COMPARATIVA FINAL (Primeras 4 rutas) ---")
final_pred = model(features, training=False) * SCALE_FACTOR
final_real = labels * SCALE_FACTOR

for i in range(min(4, len(final_real))):
    p = final_pred.numpy().flatten()[i]
    r = final_real.numpy().flatten()[i]
    diff = abs(p - r)
    print(f"Ruta {i}: Pred: {p:.4f} µs | Real: {r:.4f} µs | Error: {diff:.4f} µs")

if loss < 10:
    print("\n✅ EL MODELO MEMORIZA: La arquitectura y el flujo de datos son correctos.")
else:
    print("\n❌ EL MODELO NO CONVERGE: Revisa la normalización de entrada o la capa final.")

[*] Cargando muestra de depuración...


2026-04-22 14:22:56.676714: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:907] Skipping loop optimization for Merge node with control input: Assert_1/AssertGuard/branch_executed/_179


Iteración  0 | Loss: 138.2974 | Grad Medio: 7.39692974
Iteración 10 | Loss:  30.1566 | Grad Medio: 7.59560251
Iteración 20 | Loss:   4.8515 | Grad Medio: 7.04796648
Iteración 30 | Loss:  14.9291 | Grad Medio: 6.76532221
Iteración 40 | Loss:   1.5937 | Grad Medio: 2.70374441
Iteración 50 | Loss:   2.1944 | Grad Medio: 2.66534734

--- COMPARATIVA FINAL (Primeras 4 rutas) ---


2026-04-22 14:23:06.287134: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:907] Skipping loop optimization for Merge node with control input: Assert/AssertGuard/branch_executed/_1017


Ruta 0: Pred: 1.5565 µs | Real: 1.8630 µs | Error: 0.3064 µs
Ruta 1: Pred: 2.6508 µs | Real: 2.6605 µs | Error: 0.0097 µs
Ruta 2: Pred: 2.4728 µs | Real: 2.6581 µs | Error: 0.1853 µs
Ruta 3: Pred: 2.5154 µs | Real: 2.6854 µs | Error: 0.1701 µs

✅ EL MODELO MEMORIZA: La arquitectura y el flujo de datos son correctos.


In [5]:
#!/usr/bin/env python3
import json
import os
import numpy as np
import tensorflow as tf
import sys
import glob as glob
import pickle
import random

# Configuración de logs de TF
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Importaciones personalizadas (Asegúrate de que las rutas sean correctas)
sys.path.append("../")
from data_generatorHQoS import input_fn
from delay_model_HQoS import RouteNet_Fermi

# =========================================================
# REPRODUCIBILIDAD
# =========================================================
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# =========================================================
# CONFIGURACIÓN DE FEATURES
# =========================================================
FEATURE_KEYS = [
    'traffic', 'packets', 'eq_lambda', 'avg_pkts_lambda', 'exp_max_factor',
    'pkts_lambda_on', 'avg_t_off', 'avg_t_on', 'ar_a', 'sigma',
    'queue_size', 'policy', 'priority', 'weight', 'path_to_queue'
]

CAPACITY_KEY = "capacity"

# =========================================================
# FUNCIONES DE APOYO PARA DATASET
# =========================================================

def make_key(inputs):
    """
    Genera un hash único basado en las features de configuración.
    Maneja RaggedTensors y tensores de múltiples dimensiones.
    """
    parts = []
    for k in FEATURE_KEYS:
        v = inputs[k]
        
        # 1. Manejo específico para RaggedTensors (evita el error de len())
        if isinstance(v, tf.RaggedTensor):
            v_flat = v.flat_values
            v_flat = tf.reshape(v_flat, [-1])
        else:
            # 2. Para Tensores normales, aplanamos a 1D
            v_flat = tf.reshape(v, [-1])
        
        # 3. Convertir a string y unir valores por coma
        feat_str = tf.strings.reduce_join(tf.as_string(v_flat), separator=",")
        parts.append(feat_str)

    # Unir todas las features con un separador de tubería
    return tf.strings.reduce_join(parts, separator="|")


def build_min_max_dataset(ds):
    """
    Filtra el dataset para quedarse solo con las muestras de 
    mínima y máxima capacidad por cada configuración única.
    """
    min_store = {}
    max_store = {}

    print("[*] Analizando dataset para filtrar capacidades extremas...")
    
    for inputs, y in ds:
        # Generamos la clave (usando .numpy() porque estamos en modo Eager aquí)
        key = make_key(inputs).numpy()
        
        # Obtenemos la capacidad media de la muestra
        cap = tf.reduce_mean(inputs[CAPACITY_KEY]).numpy()

        if key not in min_store:
            min_store[key] = (cap, (inputs, y))
            max_store[key] = (cap, (inputs, y))
        else:
            # Actualizar si es menor que el mínimo conocido
            if cap < min_store[key][0]:
                min_store[key] = (cap, (inputs, y))
            # Actualizar si es mayor que el máximo conocido
            if cap > max_store[key][0]:
                max_store[key] = (cap, (inputs, y))

    # Reconstruir lista de datos
    all_data = []
    for k in min_store:
        all_data.append(min_store[k][1])
        all_data.append(max_store[k][1])

    print(f"[*] Filtrado completado. Muestras finales: {len(all_data)}")

    return tf.data.Dataset.from_generator(
        lambda: (x for x in all_data),
        output_signature=ds.element_spec
    )

# =========================================================
# PARÁMETROS Y RUTAS
# =========================================================
LEARNING_RATE = 0.001
TRAIN_PATH = "../final_set_dataset/oran-hqos_merged_ALL/train"
TEST_PATH = "../final_set_dataset/oran-hqos_merged_ALL/validation"
CKPT_DIR = "./HQoS_checkpoints_test_normal_ALL_DATASETv2EXP/"
ZSCORE_PATH = "../final_set_dataset/oran-hqos_merged_ALL/zscore_stats.json"

BATCH_SIZE = 8
EPOCHS = 500

os.makedirs(CKPT_DIR, exist_ok=True)
LOG_CSV = os.path.join(CKPT_DIR, "learning_curve02.csv")

# =========================================================
# CARGA Y PROCESAMIENTO DE DATASETS
# =========================================================
print("[1] Cargando y transformando dataset de TRAIN...")
ds_raw = input_fn(TRAIN_PATH, shuffle=True)
ds_train = build_min_max_dataset(ds_raw)
ds_train = ds_train.shuffle(buffer_size=100)

print("[2] Cargando y transformando dataset de EVAL...")
ds_eval_raw = input_fn(TEST_PATH, shuffle=False)
ds_eval = build_min_max_dataset(ds_eval_raw)

# Calcular número de pasos
num_samples = 0
for _ in ds_train:
    num_samples += 1

ds_train = ds_train.repeat()
steps_per_epoch = max(1, num_samples // BATCH_SIZE)

# =========================================================
# MODELO Y Z-SCORE
# =========================================================
model = RouteNet_Fermi()

def update_model_z_score(model, z_score_dict):
    if not hasattr(model, 'z_score') or model.z_score is None:
        model.z_score = {}
    model.z_score.update(z_score_dict)

if os.path.exists(ZSCORE_PATH):
    with open(ZSCORE_PATH, 'r') as f:
        loaded_z_score = json.load(f)
    update_model_z_score(model, loaded_z_score)
    print("[*] Z-Score cargado correctamente.")
else:
    print("[!] ERROR: No se encontró el archivo Z-Score.")

# =========================================================
# REANUDACIÓN DE CHECKPOINT
# =========================================================
checkpoints = glob.glob(os.path.join(CKPT_DIR, "epoch_*_loss_*.index"))
if checkpoints:
    checkpoints.sort(key=lambda x: float(x.split('loss_')[1].split('.index')[0]))
    best_checkpoint = checkpoints[0].replace('.index', '')
    
    # "Build" dummy para inicializar variables antes de cargar pesos
    for x_dummy, _ in ds_eval.take(1):
        _ = model(x_dummy, training=False)
    
    model.load_weights(best_checkpoint).expect_partial()
    print(f"[*] Pesos cargados desde: {best_checkpoint}")
else:
    print("[*] Iniciando entrenamiento desde cero.")

# =========================================================
# COMPILACIÓN Y ENTRENAMIENTO
# =========================================================
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_object = tf.keras.losses.MeanAbsolutePercentageError()
model.compile(loss=loss_object, optimizer=optimizer)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CKPT_DIR, "epoch_{epoch:02d}_loss_{val_loss:.4f}"),
        save_best_only=True, save_weights_only=True, monitor='val_loss'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.CSVLogger(LOG_CSV, append=True),
    tf.keras.callbacks.TerminateOnNaN()
]

print(f"\nIniciando fit: {num_samples} muestras, {steps_per_epoch} pasos por epoch.")
history = model.fit(
    ds_train,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=ds_eval,
    callbacks=callbacks
)

# Guardar historial
with open(os.path.join(CKPT_DIR, 'history_final.pkl'), 'wb') as f:
    pickle.dump(history.history, f)

[1] Cargando y transformando dataset de TRAIN...
[*] Analizando dataset para filtrar capacidades extremas...
[*] Filtrado completado. Muestras finales: 1558
[2] Cargando y transformando dataset de EVAL...
[*] Analizando dataset para filtrar capacidades extremas...
[*] Filtrado completado. Muestras finales: 940
[*] Z-Score cargado correctamente.
[*] Iniciando entrenamiento desde cero.

Iniciando fit: 1558 muestras, 194 pasos por epoch.
Epoch 1/500


/home/administrator/6gsenses/venv/lib/python3.7/site-packages/tensorflow/python/framework/indexed_slices.py:449: UserWarning: Converting sparse IndexedSlices(IndexedSlices(indices=Tensor("gradients/RaggedTile_1/GatherV2_1_grad/Reshape_1:0", shape=(None,), dtype=int64), values=Tensor("gradients/RaggedTile_1/GatherV2_1_grad/Reshape:0", shape=(None, 1), dtype=float32), dense_shape=Tensor("gradients/RaggedTile_1/GatherV2_1_grad/Cast:0", shape=(2,), dtype=int32))) to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "shape. This may consume a large amount of memory." % value)
/home/administrator/6gsenses/venv/lib/python3.7/site-packages/tensorflow/python/framework/indexed_slices.py:449: UserWarning: Converting sparse IndexedSlices(IndexedSlices(indices=Tensor("gradients/RaggedGetItem_1/GatherV2_grad/Reshape_1:0", shape=(None,), dtype=int64), values=Tensor("gradients/RaggedGetItem_1/GatherV2_grad/Reshape:0", shape=(None, 32), dtype=float32), dense_shape=Tensor("gr

194/194 [==============================] - 46s 104ms/step - loss: 20.4535 - val_loss: 12.9181
Epoch 2/500
194/194 [==============================] - 19s 97ms/step - loss: 20.8487 - val_loss: 15.2255
Epoch 3/500
194/194 [==============================] - 18s 95ms/step - loss: 21.9582 - val_loss: 13.1022
Epoch 4/500
194/194 [==============================] - 18s 93ms/step - loss: 19.9201 - val_loss: 13.1237
Epoch 5/500
194/194 [==============================] - 18s 92ms/step - loss: 21.1858 - val_loss: 16.1034
Epoch 6/500
194/194 [==============================] - 18s 94ms/step - loss: 20.7686 - val_loss: 15.6354
Epoch 7/500
194/194 [==============================] - 27s 138ms/step - loss: 19.2767 - val_loss: 12.6237
Epoch 8/500
194/194 [==============================] - 16s 85ms/step - loss: 21.0295 - val_loss: 12.6067
Epoch 9/500
194/194 [==============================] - 17s 90ms/step - loss: 19.0737 - val_loss: 11.5351
Epoch 10/500
194/194 [==============================] - 18s 91ms/